# Solution B2 · Choosing the threshold with the real costs

**Type**: Implement with scikit-learn

## Objective

Given a cost for each kind of error (an unpaid loan costs 10, a good customer turned down costs 1), choose the decision threshold **using only the training set**
(out-of-sample predictions with `cross_val_predict`), compare it with the theoretical threshold $c_{FP}/(c_{FP}+c_{FN})$ and with `class_weight="balanced"`, and measure the cost on the test set.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

df = ds.loans(with_missing=False)
df["ratio"] = df["amount"] / df["income"]
cols = ["ratio", "age", "job_tenure", "income"]
X_tr, X_te, y_tr, y_te = train_test_split(df[cols], df["defaulted"], test_size=0.3, stratify=df["defaulted"], random_state=0)
y_tr, y_te = y_tr.to_numpy(), y_te.to_numpy()
cv = StratifiedKFold(5, shuffle=True, random_state=0)
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
thresholds = np.round(np.arange(0.05, 0.951, 0.01), 2)

## Commented solution

In [ ]:
def cost(y, p, threshold):
    reject = p >= threshold
    return int(10 * ((y == 1) & ~reject).sum() + 1 * ((y == 0) & reject).sum())

In [ ]:
oof = cross_val_predict(model, X_tr, y_tr, cv=cv, method="predict_proba")[:, 1]         # predictions never seen by the model that produces them
oof_costs = np.array([cost(y_tr, oof, t) for t in thresholds])
best_threshold = float(thresholds[np.argmin(oof_costs)])
theoretical_threshold = 1 / 11
print(f"minimum-cost threshold (training set, out-of-sample): {best_threshold} | theoretical: {theoretical_threshold:.3f}")
print("the five best:", thresholds[np.argsort(oof_costs)[:5]])

In [ ]:
model.fit(X_tr, y_tr)
p_te = model.predict_proba(X_te)[:, 1]
balanced = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced")).fit(X_tr, y_tr)
test_costs = {
    "threshold 0.5": cost(y_te, p_te, 0.5),
    "best threshold": cost(y_te, p_te, best_threshold),
    "theoretical threshold": cost(y_te, p_te, theoretical_threshold),
    "class_weight balanced (0.5)": cost(y_te, balanced.predict_proba(X_te)[:, 1], 0.5),
    "approve all": cost(y_te, p_te, 1.1),
    "reject all": cost(y_te, p_te, 0.0),
}
saving = 1 - test_costs["best threshold"] / test_costs["threshold 0.5"]
print(pd.Series(test_costs).to_string())
print(f"saving compared to threshold 0.5: {saving:.0%}")

## Check

In [ ]:
with verify():
    check(cost(np.array([1, 0]), np.array([0.2, 0.9]), 0.5) == 11, "cost: a defaulter not caught (10) + a good one turned down (1)", "FN weighs 10, FP weighs 1")
    check(oof is not None and oof.shape == (len(y_tr),) and len(oof_costs) == len(thresholds), "out-of-sample predictions and one cost per threshold", "predict_proba[:, 1] from cross_val_predict")
    check(0.05 <= best_threshold <= 0.2 and abs(best_threshold - theoretical_threshold) < 0.06, "the best threshold is close to the theoretical 1/11 (about 0.09)", "with calibrated probabilities the best threshold is c_FP / (c_FP + c_FN)")
    check(test_costs["best threshold"] < 0.7 * test_costs["threshold 0.5"], "with the best threshold the cost drops by over 30% compared to 0.5", "use the threshold found on the training set, applied to the test set")
    check(test_costs["best threshold"] < test_costs["class_weight balanced (0.5)"] and test_costs["best threshold"] < min(test_costs["approve all"], test_costs["reject all"]),
          "it beats class_weight balanced and the two trivial policies", "compare the six costs")
    check(close(saving, 1 - test_costs["best threshold"] / test_costs["threshold 0.5"]) and saving > 0.3, "correct saving", "1 - best cost / cost at 0.5")
    check(test_costs["threshold 0.5"] > test_costs["reject all"], "at threshold 0.5 the model costs more than rejecting everyone: the default threshold can lose value", "compare threshold 0.5 and reject all")
done()

## What you learned

- With asymmetric costs the right threshold is far from 0.5: here about **0.09**, which coincides with the theoretical rule $c_{FP}/(c_{FP}+c_{FN}) = 1/11$. It is a
  general result: if the probabilities are calibrated, you should act whenever the expected cost of not acting exceeds that of acting.
- The threshold is chosen on the **training set** with out-of-sample probabilities (`cross_val_predict`), and the test set evaluates it once. Choosing it on the test set
  would be optimizing on the grade that judges you.
- `class_weight="balanced"` moves the model in the right direction but "blindly": it does not know the costs (here 10 to 1, not the ratio between the classes). The cost you get
  is intermediate. The explicit threshold is more flexible and leaves the probabilities calibrated.
- At threshold 0.5 the model costs **more** than rejecting all the applicants: a model "with AUC 0.78" can lose money if the threshold is wrong. Always compare with
  the trivial policies.

## Common pitfall

Choosing the threshold by looking at the ROC curve "at the point closest to the top left corner": it is the threshold that assumes equal costs and 50% prevalence. With
real costs and prevalences the right point of the curve is another one.

## Interview question

> *Click on a question to show its answer.*

<details><summary>How do you choose the threshold of a classifier in production?</summary>

I start from the costs (or from the operational constraint: for example "I can check at most 100 cases a day"), not from the metric. With calibrated probabilities the theoretical threshold
is c_FP/(c_FP+c_FN); in practice I search for it on a grid with out-of-sample predictions on the training set and validate it once on the test set. I recheck it when the
prevalence or the costs change, because the threshold depends on both.

</details>